In [1]:
import requests

url = "http://rest.kegg.jp/list/pathway/hsa"
response = requests.get(url)

for line in response.text.strip().split("\n"):
    entry, description = line.split("\t")
    pathway_id = entry  # e.g., hsa00010
    print(pathway_id, "-", description)

hsa01100 - Metabolic pathways - Homo sapiens (human)
hsa01200 - Carbon metabolism - Homo sapiens (human)
hsa01210 - 2-Oxocarboxylic acid metabolism - Homo sapiens (human)
hsa01212 - Fatty acid metabolism - Homo sapiens (human)
hsa01230 - Biosynthesis of amino acids - Homo sapiens (human)
hsa01232 - Nucleotide metabolism - Homo sapiens (human)
hsa01250 - Biosynthesis of nucleotide sugars - Homo sapiens (human)
hsa01240 - Biosynthesis of cofactors - Homo sapiens (human)
hsa01320 - Sulfur cycle - Homo sapiens (human)
hsa00010 - Glycolysis / Gluconeogenesis - Homo sapiens (human)
hsa00020 - Citrate cycle (TCA cycle) - Homo sapiens (human)
hsa00030 - Pentose phosphate pathway - Homo sapiens (human)
hsa00040 - Pentose and glucuronate interconversions - Homo sapiens (human)
hsa00051 - Fructose and mannose metabolism - Homo sapiens (human)
hsa00052 - Galactose metabolism - Homo sapiens (human)
hsa00053 - Ascorbate and aldarate metabolism - Homo sapiens (human)
hsa00500 - Starch and sucrose met

In [8]:
import pyodbc

try:
    sql_conn = pyodbc.connect(
        "DRIVER={SQL Server};"
        "SERVER=DESKTOP-9TQJR1G\\SQLEXPRESS;"
        "DATABASE=GeneDataBase;"
        "Trusted_Connection=yes;"
    )
    cursor = sql_conn.cursor()
    print("Successfully connected to the database!")
except pyodbc.Error as e:
    print("Error connecting to SQL Server:", e)

Successfully connected to the database!


In [ ]:
cursor.execute("""
IF NOT EXISTS (
    SELECT * FROM sysobjects WHERE name='PathwayIds' AND xtype='U'
)
CREATE TABLE PathwayIds (
    pathway_id VARCHAR(20) PRIMARY KEY,
    name TEXT
)
""")
sql_conn.commit()

In [ ]:
import pyodbc
import requests

# Step 1: Fetch KEGG Pathway data
url = "http://rest.kegg.jp/list/pathway/hsa"
response = requests.get(url)

pathway_data = []
for line in response.text.strip().split("\n"):
    entry, description = line.split("\t")
    pathway_id = entry  # e.g., hsa00010
    pathway_data.append((pathway_id, description))



# Step 3: Create a temporary table
cursor.execute("""
IF OBJECT_ID('tempdb..#TempPathwayIds') IS NOT NULL DROP TABLE #TempPathwayIds;
CREATE TABLE #TempPathwayIds (
    pathway_id VARCHAR(20) PRIMARY KEY,
    name TEXT
);
""")

# Step 4: Insert pathway data into the temp table in batch
insert_sql = "INSERT INTO #TempPathwayIds (pathway_id, name) VALUES (?, ?);"
cursor.executemany(insert_sql, pathway_data)

# Step 5: Merge temp table into main table
cursor.execute("""
MERGE INTO PathwayIds AS target
USING #TempPathwayIds AS source
ON target.pathway_id = source.pathway_id
WHEN MATCHED THEN
    UPDATE SET target.name = source.name
WHEN NOT MATCHED THEN
    INSERT (pathway_id, name)
    VALUES (source.pathway_id, source.name);
""")

# Step 6: Commit transaction
sql_conn.commit()

# Step 7: Optional cleanup (not necessary, temp table is session-scoped)
# cursor.execute("DROP TABLE #TempPathwayIds")

# Close connection
cursor.close()
conn.close()


In [ ]:
import pyodbc

import requests

url = "http://rest.kegg.jp/list/pathway/hsa"
response = requests.get(url)

pathway_data = []
for line in response.text.strip().split("\n"):
    entry, description = line.split("\t")
    pathway_id = entry  # e.g., hsa00010
    pathway_data.append((pathway_id, description))

conn = pyodbc.connect("DRIVER={SQL Server};SERVER=localhost;DATABASE=MyDB;Trusted_Connection=yes;")
cursor = conn.cursor()

for row in pathway_data:
    cursor.execute("""
    MERGE INTO PathwayIds AS target
    USING (SELECT ? AS pathway_id, ? AS name) AS source
    ON target.pathway_id = source.pathway_id
    WHEN MATCHED THEN
        UPDATE SET name = source.name
    WHEN NOT MATCHED THEN
        INSERT (pathway_id, name)
        VALUES (source.pathway_id, source.name);
    """, row)

conn.commit()


In [ ]:
def translate_compounds(self, compounds: list):
        compound_url_base = self.base_url + "/get/"
        parsed_compounds = []
        for compound_batch_idx in range(0, len(compounds), 20):
            compound_batch = compounds[compound_batch_idx:compound_batch_idx + 20]
            compounds_str = "+".join(f"cpd:{cid}" for cid in compound_batch)
            send_url = compound_url_base + compounds_str

            print(f"Fetching KEGG data for compounds: {compounds_str}")
            response = requests.get(send_url)

            if response.ok:
                print(f"{GREEN}Successfully fetched KEGG data for batch: {compounds_str}{RESET}")

                entries = response.text.strip().split("///")
                for entry_text in entries:
                    entry_text = entry_text.strip()
                    if not entry_text:
                        continue

                    parsed = self.parse_kegg_entry(entry_text)
                    print(f"Parsed entry for {parsed.get('ENTRY')}:")
                    parsed_compounds.append(parsed)
            else:
                print(f"{RED}Failed to fetch batch: {compounds_str} — Status: {response.status_code}{RESET}")
        print(f"{GREEN}Finished processing all compound batches.{RESET}")
        return parsed_compounds

In [ ]:
def parse_kegg_entry(self, entry_text):
        import re
        parsed = {}
        current_key = None

        for line in entry_text.splitlines():
            # Convert multiple spaces to a single tab
            line = re.sub(r' {2,}', '\t', line)
            parts = line.split('\t')

            # Skip malformed lines
            if not parts:
                continue

            # If this line starts a new field
            if parts[0].strip():
                current_key = parts[0].strip()

            if current_key == 'ENTRY':
                parsed['ENTRY'] = parts[1].strip() if len(parts) > 1 else None
                parsed['LABEL'] = parts[2].strip() if len(parts) > 2 else None
            if current_key == 'NAME':
                if 'NAME' not in parsed and len(parts) > 1:
                    parsed['NAME'] = parts[1].strip(" ;")
                elif len(parts) > 1:
                    parsed.setdefault('SYNONYMS', []).append(parts[1].strip(" ;"))

            elif current_key in ['PATHWAY', 'MODULE', 'NETWORK'] and len(parts) >= 3:
                parsed.setdefault(current_key, {})[parts[1].strip()] = parts[2].strip()

            elif current_key == 'DBLINKS' and len(parts) > 1:
                parsed.setdefault('DBLINKS', {})
                if ": " in parts[1]:
                    code, info = parts[1].split(": ", 1)
                    parsed['DBLINKS'][code.strip()] = info.strip()

        return parsed

In [3]:
def parse_map_sql(entry_map, pathway_id):
    entity_table = []
    interactions_table = []
    reaction_table = []
    reaction_ptable = []
    seen_reactions = set()

    def clean_id(identifier):
        return identifier.split(":")[1] if ":" in identifier else identifier

    for entry_id, entry in entry_map.items():
        entity_type = entry.get("entity_type", "")
        entities = entry.get("entities", [])
        reaction_ids = entry.get("reaction", [])
        substrates = entry.get("substrates", [])
        products = entry.get("products", [])
        relation_targets = entry.get("relations", [])
        relation_types = entry.get("relation_types", [])
        reaction_type = entry.get("reaction_type", None)

        for entity in entities:
            entity_code = clean_id(entity)
            entity_table.append((entity_code, "pathway" if entity_type == "map" else entity_type))

            # Handle interactions
            for idx, targets in enumerate(relation_targets):
                for target in targets:
                    interactions_table.append((
                        entity_code,
                        clean_id(target),
                        relation_types[idx],
                        pathway_id
                    ))

            # Handle reaction participants
            for reaction_id_raw in reaction_ids or []:
                reaction_id = clean_id(reaction_id_raw)

                if entity_type == "ortholog":
                    reaction_ptable.append((reaction_id, entity_code, "ortholog"))
                    continue

                for s in substrates:
                    reaction_ptable.append((reaction_id, clean_id(s), "substrate"))
                for p in products:
                    reaction_ptable.append((reaction_id, clean_id(p), "product"))

                if reaction_id not in seen_reactions:
                    reaction_table.append((reaction_id, reaction_type, pathway_id))
                    seen_reactions.add(reaction_id)

    return entity_table, interactions_table, reaction_table, reaction_ptable


In [4]:
from bs4 import BeautifulSoup
import requests
import pprint

def pathway_parse(pathway_code):
    import requests
    from bs4 import BeautifulSoup

    url = f"http://rest.kegg.jp/get/{pathway_code}/kgml"
    response = requests.get(url)

    if not response.ok:
        raise Exception(f"Failed to fetch data for {pathway_code}")

    print(f"Successfully fetched KEGG pathway data for {pathway_code}")
    soup = BeautifulSoup(response.content, "xml")

    entry_map = {}
    for entry in soup.find_all("entry"):
        entry_id = entry.get("id")
        entry_map[entry_id] = {
            "entities": entry.get("name", "").split(),
            "reaction": entry.get("reaction", "").split() if entry.get("reaction") else [],
            "entity_type": entry.get("type")
        }

    for reaction in soup.find_all("reaction"):
        rid = reaction.get("id")
        entry = entry_map.get(rid, {})
        entry["substrates"] = [s.get("name") for s in reaction.find_all("substrate")]
        entry["products"] = [p.get("name") for p in reaction.find_all("product")]
        entry["reaction_type"] = reaction.get("type")
        entry_map[rid] = entry  # in case it wasn’t already in the map

    for rel in soup.find_all("relation"):
        src = rel.get("entry1")
        tgt = rel.get("entry2")
        if src in entry_map and tgt in entry_map:
            entry_map[src].setdefault("relations", []).append(entry_map[tgt]['entities'])
            subtype = rel.find("subtype")
            entry_map[src].setdefault("relation_types", []).append(subtype.get("name") if subtype else "unknown")

    return (*parse_map_sql(entry_map, pathway_code), entry_map)



In [5]:
entity_table, intrx_table, rx_table, rx_ptable, entry_map = pathway_parse("hsa00010")


Successfully fetched KEGG pathway data for hsa00010


In [6]:
entry_map

{'18': {'entities': ['hsa:226', 'hsa:229', 'hsa:230'],
  'reaction': ['rn:R01068'],
  'entity_type': 'gene',
  'substrates': ['cpd:C00354'],
  'products': ['cpd:C00118', 'cpd:C00111'],
  'reaction_type': 'reversible',
  'relations': [['hsa:2203', 'hsa:8789'],
   ['hsa:5211', 'hsa:5213', 'hsa:5214'],
   ['hsa:7167'],
   ['hsa:2597', 'hsa:26330'],
   ['path:hsa00030']],
  'relation_types': ['compound',
   'compound',
   'compound',
   'compound',
   'compound']},
 '42': {'entities': ['hsa:217', 'hsa:219', 'hsa:223', 'hsa:224', 'hsa:501'],
  'reaction': ['rn:R00710'],
  'entity_type': 'gene',
  'substrates': ['cpd:C00084'],
  'products': ['cpd:C00033'],
  'reaction_type': 'reversible',
  'relations': [['hsa:124',
    'hsa:125',
    'hsa:126',
    'hsa:127',
    'hsa:128',
    'hsa:130',
    'hsa:131'],
   ['hsa:10327']],
  'relation_types': ['compound', 'compound']},
 '43': {'entities': ['ko:K01905', 'ko:K22224', 'ko:K24012'],
  'reaction': ['rn:R00229'],
  'entity_type': 'ortholog'},
 '4

In [95]:
entity_table

[('226', 'gene'),
 ('229', 'gene'),
 ('230', 'gene'),
 ('217', 'gene'),
 ('219', 'gene'),
 ('223', 'gene'),
 ('224', 'gene'),
 ('501', 'gene'),
 ('K01905', 'ortholog'),
 ('K22224', 'ortholog'),
 ('K24012', 'ortholog'),
 ('218', 'gene'),
 ('221', 'gene'),
 ('222', 'gene'),
 ('C00033', 'compound'),
 ('hsa00030', 'pathway'),
 ('hsa00500', 'pathway'),
 ('K01568', 'ortholog'),
 ('10327', 'gene'),
 ('124', 'gene'),
 ('125', 'gene'),
 ('126', 'gene'),
 ('127', 'gene'),
 ('128', 'gene'),
 ('130', 'gene'),
 ('131', 'gene'),
 ('K01568', 'ortholog'),
 ('5160', 'gene'),
 ('5161', 'gene'),
 ('5162', 'gene'),
 ('5160', 'gene'),
 ('5161', 'gene'),
 ('5162', 'gene'),
 ('1737', 'gene'),
 ('160287', 'gene'),
 ('3939', 'gene'),
 ('3945', 'gene'),
 ('3948', 'gene'),
 ('92483', 'gene'),
 ('hsa00620', 'pathway'),
 ('5313', 'gene'),
 ('5315', 'gene'),
 ('hsa00020', 'pathway'),
 ('2023', 'gene'),
 ('2026', 'gene'),
 ('2027', 'gene'),
 ('387712', 'gene'),
 ('441531', 'gene'),
 ('5223', 'gene'),
 ('5224', 'gene

In [93]:
rx_ptable[0:5]

[('R01068', 'C00354', 'substrate'),
 ('R01068', 'C00118', 'product'),
 ('R01068', 'C00111', 'product'),
 ('R01068', 'C00354', 'substrate'),
 ('R01068', 'C00118', 'product')]

In [94]:
intrx_table[0:5]

[('226', '2203', 'compound', 'hsa00010'),
 ('226', '8789', 'compound', 'hsa00010'),
 ('226', '5211', 'compound', 'hsa00010'),
 ('226', '5213', 'compound', 'hsa00010'),
 ('226', '5214', 'compound', 'hsa00010')]